<a href="https://colab.research.google.com/github/SPkit07/Sale/blob/main/%E0%B8%A2%E0%B8%AD%E0%B8%94%E0%B8%82%E0%B8%B2%E0%B8%A2%20power%20bi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install msoffcrypto-tool python-calamine

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.1/935.1 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.6/114.6 kB 7.3 MB/s eta 0:00:00


In [3]:
!pip install python-calamine

In [4]:
import msoffcrypto
import pandas as pd
import io
import openpyxl

In [5]:
pd.set_option('display.max_rows', None)

In [6]:
from google.colab import drive

# ขั้นตอนที่ 1: ยกเลิกการเชื่อมต่อ Drive ปัจจุบัน
drive.flush_and_unmount()

# ***เมื่อเสร็จสิ้นขั้นตอนการยกเลิกการเชื่อมต่อแล้ว***
# ***ให้รันเซลล์ต่อไปเพื่อเชื่อมต่อ Drive ด้วยบัญชีใหม่***

Drive not mounted, so nothing to flush and unmount.


In [8]:
from google.colab import drive

# ขั้นตอนที่ 2: รันเซลล์นี้เพื่อเชื่อมต่อ Drive ด้วยบัญชีใหม่
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# 1. กำหนดชื่อไฟล์และรหัสผ่าน
file_path = '/content/drive/My Drive/ทดลองเชื่อม กูเกิลไดรฟ์/ยอดขายรวมทุกสาขาBplus2569.xlsx'
password = '18651865'

In [ ]:
data = []
month_names = ['ม.ค.69', 'ก.พ.69', 'มี.ค.69', 'เม.ย.69', 'พ.ค.69', 'มิ.ย.69',
               'ก.ค.69', 'ส.ค.69', 'ก.ย.69', 'ต.ค.69', 'พ.ย.69', 'ธ.ค.69']
# 2. สร้างออบเจ็กต์สำหรับจัดการไฟล์
temp_file = io.BytesIO()

file_content = None # Initialize file_content outside try block
try:
    with open(file_path, 'rb') as f:
        file_content = f.read() # Reads *all* content

    # Use a copy of the content for msoffcrypto-tool to avoid issues with stream position
    f_for_decrypt = io.BytesIO(file_content)

    # Attempt to decrypt
    office_file = msoffcrypto.OfficeFile(f_for_decrypt)
    office_file.load_key(password=password)
    office_file.decrypt(temp_file)
    print("File decrypted successfully.")
except msoffcrypto.exceptions.DecryptionError as e:
    if "Document is not encrypted" in str(e):
        print("Document is not encrypted. Reading original file content directly.")
        # If not encrypted, just write the original content to temp_file
        if file_content is not None:
            temp_file.write(file_content)
        else:
            print("Error: file_content was not read.")
            raise e # Re-raise if file_content is unexpectedly empty
    else:
        raise e # Re-raise other decryption errors
except FileNotFoundError:
    print(f"Error: File not found at {file_path}. Please check the path and filename again.")
    raise
except Exception as e:
    print(f"An unexpected error occurred during file handling: {e}")
    raise e

# Reset temp_file's cursor to the beginning for subsequent reading by pandas
temp_file.seek(0)

# Diagnostic: Print magic bytes to identify file type
magic_bytes = temp_file.read(4) # Read the first 4 bytes
temp_file.seek(0) # Reset cursor again after reading magic bytes
print(f"File Magic Bytes: {magic_bytes!r}")

In [ ]:
cols = 'A:H,T'
# 3. ใช้ Pandas อ่านไฟล์จากหน่วยความจำ
for i in month_names:
    df = pd.read_excel(temp_file , sheet_name= i ,usecols=cols, header= 4 ,engine= 'openpyxl')
    df = df.iloc[0:35,:]
    # แปลงข้อมูลเป็นวันที่
    df['DATE'] = pd.to_datetime(df['Unnamed: 0'], dayfirst=True ,errors='coerce')
    #ลบค่าว่าง
    df.dropna(subset='DATE',inplace=True)
    #กรอกวันที่ที่มากกว่า 2020
    df = df[df['DATE'].dt.year > 2020]

    df['DATE'] = df['DATE'].dt.date

    df.drop(columns=['Unnamed: 0'] , inplace=True)

    df.rename(columns={'Unnamed: 19': 'WH'}, inplace=True)

    data.append(df)

full = pd.concat(data , ignore_index=True)
# ลบซ้ำ
full.drop_duplicates(inplace=True)

full.insert(0 ,'DATE1',full['DATE'])

full.drop(columns='DATE',inplace=True)

full.rename(columns={'DATE1':'DATE'},inplace=True)

